# 🌸 Orchid 1.0 — Primer LLM Colombiano

**First Colombian LLM** — 2B ternary-weight model built on [BitNet b1.58](https://huggingface.co/microsoft/bitnet-b1.58-2B-4T), aligned with ORPO, served by [ternative.cpp](https://github.com/michelangeloromerochisco/orchid-1.0).

Model: [MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0) · License: Apache 2.0

---

### Instructions

1. **Runtime → Change runtime type → CPU** (free, no time limit)
2. **Runtime → Run all** (`Ctrl+F9`)
3. Wait for the server to start:
   - **First run**: ~8–10 min (merges LoRA into model, saves cache)
   - **Subsequent runs** (same session, no restart): ~30 seconds
4. Click the **public Gradio URL** at the bottom

> **Speed**: ~6–8 tok/s with AVX2 + OpenBLAS.
> The public URL stays valid for 72 hours while the session is active.
> Do **not** restart the runtime between conversations — re-run only Cell 5 + Cell 6 if needed.

In [ ]:
# ── Cell 1: Check CPU capabilities ───────────────────────────
import subprocess, os

# Show CPU model and core count
os.system('echo "CPU:" && grep "model name" /proc/cpuinfo | head -1 | cut -d: -f2')
os.system('echo "Cores:" && nproc')
os.system('echo "RAM:" && free -h | awk \'/Mem:/{print $2}\'')

# Verify AVX2 (required for ternative.cpp fast path)
flags = open('/proc/cpuinfo').read()
avx2 = 'avx2' in flags
print(f'\nAVX2: {"✓ available — fast ternary kernels active" if avx2 else "✗ not available — will be slower"}')
if not avx2:
    print('WARNING: this CPU does not support AVX2. Inference will be ~2x slower.')

In [ ]:
# ── Cell 2: Get ternative.cpp source from GitHub ─────────────
import os, sys

# Clean any previous attempt
os.system('rm -rf /content/orchid-repo /content/ternative')

print('Cloning ternative.cpp source...')
ret = os.system('git clone --depth 1 https://github.com/michelangeloromerochisco/orchid-1.0 /content/orchid-repo')
if ret != 0:
    print('Clone FAILED'); sys.exit(1)

os.system('cp -r /content/orchid-repo/ternative /content/ternative')

# Verify critical files are present
for f in ['CMakeLists.txt', 'src/model.cpp', 'cuda/ops_cuda.cu']:
    path = f'/content/ternative/{f}'
    exists = os.path.exists(path)
    print(f'{"OK" if exists else "MISSING"}  {f}')
    if not exists:
        sys.exit(1)

print('Source ready.')

In [ ]:
# ── Cell 3: Build ternative.cpp (CPU + OpenBLAS) ─────────────
import subprocess, sys, os

# Install build deps
subprocess.run(['apt-get', 'install', '-y', '-q',
                'cmake', 'libopenblas-dev', 'libgomp1'], check=True)

# Clean any stale build
os.system('rm -rf /content/ternative/build')

print('Running cmake...')
r = subprocess.run(
    ['cmake', '-B', 'build',
     '-DCMAKE_BUILD_TYPE=Release',
     '-DTERNATIVE_CUDA=OFF',       # CPU only — avoids nvcc cmake issues
     '-DTERNATIVE_BUILD_TESTS=OFF',
     '-DCMAKE_CXX_FLAGS=-O3 -march=native -fopenmp',
     '-DCMAKE_EXE_LINKER_FLAGS=-fopenmp'],
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('cmake FAILED:\n', r.stderr[-2000:])
    sys.exit(1)
print('cmake OK. Building...')

r = subprocess.run(
    ['cmake', '--build', 'build', '--parallel', '4'],
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Build FAILED:\n', r.stderr[-3000:])
    sys.exit(1)

os.system('cp /content/ternative/build/ternative /usr/local/bin/ternative')
print('Build complete! (~6-8 tok/s on Colab CPUs with AVX2)')

In [ ]:
# ── Cell 4: Download Orchid models ───────────────────────────
from huggingface_hub import hf_hub_download

os.makedirs('/content/models', exist_ok=True)
MODEL_REPO = 'MicheRomChis/orchid-1.0'

print('Downloading base model (1.1 GB)...')
hf_hub_download(MODEL_REPO, 'ggml-model-i2_s.gguf', local_dir='/content/models')
print('Downloading LoRA adapter (94 MB)...')
hf_hub_download(MODEL_REPO, 'dpo_aligned-lora.gguf', local_dir='/content/models')
print('Models ready!')
os.system('ls -lh /content/models/')

In [ ]:
# ── Cell 5: Start ternative.cpp server ───────────────────────
import subprocess, threading, time, requests as req, os, sys

PORT = 8899
CACHE_DIR = '/content/.ternative_cache'   # persists across runtime restarts

os.makedirs(CACHE_DIR, exist_ok=True)

# Hard-kill anything holding our port
os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill -9 -f ternative 2>/dev/null; sleep 3')

# Set cache dir so the built model survives runtime restarts
env = os.environ.copy()
env['TERNATIVE_CACHE_DIR'] = CACHE_DIR

proc = subprocess.Popen(
    ['ternative',
     '--model', '/content/models/ggml-model-i2_s.gguf',
     '--lora',  '/content/models/dpo_aligned-lora.gguf',
     '--server', '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)

cache_built = [False]

def _log():
    for line in iter(proc.stdout.readline, ''):
        l = line.rstrip()
        if not l: continue
        # Only print non-spammy lines
        if '[apply_lora]' not in l:
            print('[ternative]', l)
        elif not cache_built[0]:
            print('[ternative] Merging LoRA into model (first run only — ~5-8 min)...')
            cache_built[0] = True
        if 'Failed to bind' in l:
            print(f'FATAL: port {PORT} in use — try Runtime → Restart session')

threading.Thread(target=_log, daemon=True).start()

print(f'Starting on port {PORT}...')
print('First run (no cache): 5-10 min to merge LoRA.')
print('Subsequent runs (cache hit): ~30s.\n')

for i in range(600):          # 10 min max
    time.sleep(1)
    if proc.poll() is not None:
        print(f'ERROR: process exited (code {proc.returncode})')
        sys.exit(1)
    try:
        r = req.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'\nServer ready after {i+1}s  ✓')
            break
    except Exception:
        pass
    if (i+1) % 60 == 0:
        elapsed = (i+1)//60
        print(f'  {elapsed} min — still merging LoRA...')
else:
    print('Timeout (10 min). Try Runtime → Restart session and re-run all cells.')
    sys.exit(1)

In [ ]:
# ── Cell 6: Gradio chat UI ────────────────────────────────────
!pip install -q "gradio>=5.0,<6.0"

import gradio as gr, requests as req, re

PORT = 8899
EOT  = "<|eot_id|>"

SYSTEM = (
    "You are Orchid, an AI assistant created in Colombia by Michelangelo Romero Chisco. "
    "Give one short, direct answer. Match the language the user writes in."
)

CSS = """
.tribar{display:flex;height:5px;width:100%;}
.tribar span{flex:1;}
.y{background:#FCD116;}.b{background:#003893;}.r{background:#CE1126;}
"""

# ── All known garbage signals ─────────────────────────────────
_GARBAGE_RE = re.compile(
    r'\(Thinking\)'         # (Thinking) loop
    r'|\-¿'                 # Spanish punctuation artifact
    r'|¿\s*\n'              # bare ¿ + newline
    r'|\bResponse\b'        # Response: / Response in / Response is
    r'|\nUser\s*:'          # role markers on new line
    r'|\nSystem\s*:'
    r'|\nAssistant\s*:'
    r'|\nHuman\s*:'
    r'|<\|eot_id\|>'        # EOT token as literal string
    r'|\n###',
    re.IGNORECASE,
)

# Sentence splitter: punctuation OR newline
_SPLIT = re.compile(r'(?<=[.!?])\s+|\n+')


def _max_sentences(message: str) -> int:
    """More words in the question → allow more sentences in the answer."""
    w = len(message.split())
    if w <= 5:  return 2   # "hi", "how are you" → 2 sentences max
    if w <= 25: return 4   # normal question
    return 6               # long/technical question


def extract(raw: str, message: str) -> str:
    """
    Two-layer extraction:
      1. Known garbage signals → hard cut
      2. Sentence count cap → prevents fake dialogue regardless of pattern
    """
    text = raw.strip()
    if not text:
        return ""

    # Layer 1: hard cut at garbage signal
    m = _GARBAGE_RE.search(text)
    if m and m.start() > 0:
        text = text[:m.start()].strip()

    # Layer 2: sentence cap (the real root fix for fake-dialogue generation)
    limit  = _max_sentences(message)
    parts  = [s.strip() for s in _SPLIT.split(text) if s.strip()]
    capped = " ".join(parts[:limit])

    # Trim to last complete sentence so we don't end mid-word
    last = max(capped.rfind('.'), capped.rfind('!'), capped.rfind('?'))
    if 0 < last < len(capped) - 1:
        capped = capped[:last + 1]

    result = capped.strip()
    if not result:                             # safety: never return empty
        m2 = re.search(r'[.!?]', raw)
        result = raw[:m2.end()].strip() if m2 else raw[:120].strip()
    return result


# ── Prompt ────────────────────────────────────────────────────
def build_prompt(message: str, history: list) -> str:
    parts = [f"System: {SYSTEM}{EOT}"]
    for h in (history or []):
        if isinstance(h, (list, tuple)) and len(h) == 2:
            parts.append(f"User: {h[0]}{EOT}Assistant: {h[1]}{EOT}")
    parts.append(f"User: {message}{EOT}Assistant:")
    return "".join(parts)


# ── Chat ──────────────────────────────────────────────────────
def chat(message, history, temperature, max_tokens):
    if not message.strip():
        return "", history or []
    try:
        r = req.post(
            f"http://127.0.0.1:{PORT}/v1/completions",
            json={
                "model":       "orchid",
                "prompt":      build_prompt(message, history),
                "max_tokens":  max_tokens,
                "temperature": temperature,
                "stream":      False,
                "stop":        [EOT, "\nUser:"],
            },
            timeout=max_tokens * 6,
        )
        if r.ok:
            raw   = r.json()["choices"][0]["text"]
            reply = extract(raw, message)
        else:
            reply = f"Error {r.status_code}: {r.text[:200]}"
    except Exception as e:
        reply = f"Error: {e}"
    return "", (history or []) + [[message, reply]]


# ── UI ────────────────────────────────────────────────────────
with gr.Blocks(title="Orchid 1.0") as demo:
    gr.HTML(
        '<div class="tribar">'
        + '<span class="y"></span>' * 3
        + '<span class="b"></span>' * 3
        + '<span class="r"></span>' * 3
        + "</div>",
    )
    gr.Markdown("# 🌸 Orchid 1.0\n**Primer LLM Colombiano** · 2B ternary · Apache 2.0")
    chatbot = gr.Chatbot(
        height=460,
        type="tuples",
        allow_tags=False,
        label="Orchid",
    )
    with gr.Row():
        msg  = gr.Textbox(placeholder="Hola Orchid! ¿Cómo estás?", label="", scale=5)
        send = gr.Button("Enviar ▶", scale=1, variant="primary")
    with gr.Row():
        temp = gr.Slider(0.1, 1.0, value=0.3, step=0.05, label="Temperatura")
        maxt = gr.Slider(32,  128, value=80,   step=16,   label="Max tokens")
    gr.Button("Limpiar").click(lambda: (None,), None, chatbot)
    gr.Markdown(
        "---\n*[MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0)"
        " · ternative.cpp · Colombia 🇨🇴*"
    )
    msg.submit(chat, [msg, chatbot, temp, maxt], [msg, chatbot])
    send.click(chat,  [msg, chatbot, temp, maxt], [msg, chatbot])

demo.launch(share=True)